## 基本环境 · Basic setup

首次打开运行下面 3 个 cell。它们做的事:
1. 把工作目录切到所在的代码树根 (`solutions/` 或 `tutorials/`)，
   这样 `from attention.mha import ...` 这种导入能直接生效。
2. 启用 `autoreload`，编辑 .py 文件保存后 notebook 里立刻可用，不用重启 kernel。
3. 设 `LAYERNORM_TYPE=torch`，避免 CUDA-only 算子的 import 失败。

First time you open the notebook, run the 3 cells below: cd to the tree root (whichever of `solutions/` or `tutorials/` this notebook lives in), turn on autoreload, force the pure-PyTorch LayerNorm path.

In [ ]:
import os, sys

# Walk up from the notebook's CWD until we find a directory named
# `solutions` or `tutorials`. Works no matter which tree the student
# opened. 不论 notebook 位于 solutions/ 还是 tutorials/ 都能正确定位。
ROOTS = {'solutions', 'tutorials'}
if os.path.basename(os.getcwd()) not in ROOTS:
    while os.path.basename(os.getcwd()) not in ROOTS and os.getcwd() != '/':
        os.chdir('..')
    if os.path.basename(os.getcwd()) not in ROOTS:
        # Fallback: maybe we were started at the repo root.
        if os.path.isdir('tutorials'):
            os.chdir('tutorials')
        elif os.path.isdir('solutions'):
            os.chdir('solutions')

assert os.path.basename(os.getcwd()) in ROOTS, (
    f'could not locate solutions/ or tutorials/ from {os.getcwd()}')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
os.environ.setdefault('LAYERNORM_TYPE', 'torch')
print('cwd =', os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
# Folder where the confidence chapter's reference .pt files live
control_folder = 'confidence/control_values'
assert os.path.isdir(control_folder), f'missing {control_folder}'

# 第 5 章 · Confidence

推理完拿到坐标后还要给每个原子 / 每个 token 对一个**置信度估计**。AF3 的置信头输出 4 路:

| 量 | 含义 | 维度 |
|---|---|---|
| pLDDT | per-atom local-distance-difference test | per atom |
| PAE | predicted aligned error | per token pair |
| PDE | predicted distance error | per token pair |
| resolved | 可解析 / 未解析二分类 | per atom |

ConfidenceHead 整体由前面章节的零件搭起来 (内部跑一个小型 PairformerStack), 我们本章只单独验证 DistogramHead——置信度系统的入口。

## 5.1 DistogramHead (算法 1 第 17 行)

打开 `confidence/distogram_head.py`，把 `forward` 的 TODO 填好。

DistogramHead 把 pair 表示通过一个零初始化的线性层投到 64 个距离 bin 上, 再对 (i, j) 做对称化。零初始化意味着训练初期输出近似均匀分布。

In [ ]:
from confidence.distogram_head import DistogramHead
from confidence.control_values.confidence_checks import (
    c_z, no_bins, test_inputs,
    test_module_shape, test_module_forward,
)

dh = DistogramHead(c_z=c_z, no_bins=no_bins)
test_module_shape(dh, 'distogram_head', control_folder)
test_module_forward(
    dh, 'distogram_head',
    inputs=(test_inputs['z'],),
    output_names='out',
    control_folder=control_folder,
)
print('DistogramHead ✓')

## 5.2 ConfidenceHead 装配检查

ConfidenceHead 的完整 `forward` / `memory_efficient_forward` 需要一份带 atom 级索引映射的特征字典，构造测试输入过于繁琐。这里只用 shape 检查验证 `ConfidenceHead.__init__` —— 它会构造一个内部 PairformerStack + 四个分类头，任何子模块漏建、命名错、维度错都会暴露。

In [ ]:
from confidence.confidence_head import ConfidenceHead
from confidence.control_values.confidence_checks import test_module_shape

ch = ConfidenceHead(
    n_blocks=1,
    c_s=32, c_z=c_z,
    c_s_inputs=32,
    b_pae=8, b_pde=8, b_plddt=10, b_resolved=2,
    max_atoms_per_token=5,
    pairformer_dropout=0.0,
    distance_bin_start=3.25, distance_bin_end=8.25, distance_bin_step=1.25,
)
test_module_shape(ch, 'confidence_head_init', control_folder)
print('ConfidenceHead.__init__ ✓')

## 章节小结

本章你写了 DistogramHead 的 forward，并通过 shape 检查校验了 ConfidenceHead 的整体装配。完整的 `forward` / `memory_efficient_forward` 会在端到端 notebook (`model/overview.ipynb`) 里跑到。